In [ ]:
import pandas as pd
import re

# =============================
# LOAD DATA
# =============================
file_path = "input.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master")

# =============================
# CLEAN CYCLE TIME
# =============================
ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# Cycle time in minutes
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
BASE_120T_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}
EXTRA_MACHINE = "TOYO 80T 2ND"
ALL_TARGET_MACHINES = BASE_120T_MACHINES | {EXTRA_MACHINE}

AVAILABLE_TIME_MIN = 22 * 60  # 1320 minutes

# =============================
# HELPER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\- ]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# BUILD PART → MACHINE MAP
# =============================
records = []

for child, grp in master.groupby("Child Part", sort=False):

    raw = ",".join(grp["Vertical Machines"].astype(str))
    tokens = re.split(r"[,\|/\\\n]+", raw)

    machines = set()
    for t in tokens:
        m = normalize_machine(t)
        if m in BASE_120T_MACHINES:
            machines.add(m)

    # 🔥 If part is 120T-capable, add TOYO 80T 2ND
    if machines:
        machines.add(EXTRA_MACHINE)

    if not machines:
        continue

    records.append({
        "Child Part": child,
        "Possible Machines": ", ".join(sorted(machines))
    })

part_machine_df = pd.DataFrame(records)

# =============================
# BUILD PART-WISE CAPACITY TABLE
# =============================
capacity_rows = []

for _, row in part_machine_df.iterrows():
    child = row["Child Part"]
    ct = cycle_time_min.get(child)

    if not ct or ct <= 0:
        continue

    capacity = AVAILABLE_TIME_MIN / ct

    for machine in row["Possible Machines"].split(", "):
        capacity_rows.append({
            "Child Part": child,
            "Machine": machine,
            "Capacity (units/day)": int(capacity)
        })

capacity_df = pd.DataFrame(capacity_rows)

# =============================
# DISPLAY RESULTS
# =============================
print("\n========== PART → POSSIBLE MACHINES ==========\n")
display(part_machine_df)

print("\n========== PART-WISE MACHINE CAPACITY (1 DAY) ==========\n")
display(capacity_df)


In [ ]:
import pandas as pd
import re

# =============================
# LOAD DATA
# =============================
file_path = "input.xlsx"
output_file = "part_machine_capacity.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master")

# =============================
# CLEAN CYCLE TIME
# =============================
ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce")

cycle_time_min = (
    ppm.dropna(subset=["Material", "Machine"])
       .set_index("Material")["Machine"]
       .div(60)   # seconds → minutes
       .to_dict()
)

# =============================
# MACHINE GROUP (INTERCHANGEABLE)
# =============================
MACHINE_GROUP = [
    "MP-01", "MP-05", "MP-10", "MP-17", "TOYO 80T 2ND"
]

AVAILABLE_TIME_MIN = 22 * 60  # 1320 minutes

# =============================
# HELPER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\- ]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: PART → MACHINE MAP (ALL PARTS)
# =============================
part_machine_rows = []

for child, grp in master.groupby("Child Part", sort=False):

    raw = grp["Vertical Machines"].dropna()
    if raw.empty:
        continue  # skip only if no machine info at all

    tokens = re.split(r"[,\|/\\\n]+", ",".join(raw.astype(str)))
    normalized = {normalize_machine(t) for t in tokens if normalize_machine(t)}

    # If part appears on ANY machine of the group → assign ALL machines
    if normalized.intersection(MACHINE_GROUP):
        part_machine_rows.append({
            "Child Part": child,
            "Possible Machines": ", ".join(MACHINE_GROUP)
        })

part_machine_df = pd.DataFrame(part_machine_rows)

# =============================
# STEP 2: PART-WISE MACHINE CAPACITY
# =============================
capacity_rows = []

for _, row in part_machine_df.iterrows():
    child = row["Child Part"]
    ct = cycle_time_min.get(child)

    if not ct or ct <= 0:
        continue

    capacity = AVAILABLE_TIME_MIN / ct

    for machine in MACHINE_GROUP:
        capacity_rows.append({
            "Child Part": child,
            "Machine": machine,
            "Capacity (units/day)": int(capacity)
        })

capacity_df = pd.DataFrame(capacity_rows)

# =============================
# WRITE TO EXCEL (2 SHEETS)
# =============================
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    part_machine_df.to_excel(writer, sheet_name="Part_Machine_Map", index=False)
    capacity_df.to_excel(writer, sheet_name="Part_Machine_Capacity", index=False)

print(f"✅ Excel file created: {output_file}")


In [ ]:
import pandas as pd
import re

# =============================
# LOAD DATA
# =============================
file_path = "input.xlsx"
output_file = "part_machine_capacity_120T.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master")

# =============================
# CLEAN CYCLE TIME
# =============================
ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce")

# Cycle time in minutes (seconds → minutes)
cycle_time_min = (
    ppm.dropna(subset=["Material", "Machine"])
       .set_index("Material")["Machine"]
       .div(60)
       .to_dict()
)

# =============================
# CONSTANTS (120T ONLY)
# =============================
MACHINE_GROUP_120T = ["MP-01", "MP-05", "MP-10", "MP-17"]
AVAILABLE_TIME_MIN = 22 * 60  # 1320 minutes

# =============================
# HELPER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\- ]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: PART → MACHINE MAP (120T)
# =============================
part_machine_rows = []

for child, grp in master.groupby("Child Part", sort=False):

    raw = grp["Vertical Machines"].dropna()
    if raw.empty:
        continue

    tokens = re.split(r"[,\|/\\\n]+", ",".join(raw.astype(str)))
    normalized = {normalize_machine(t) for t in tokens if normalize_machine(t)}

    # If part appears on ANY 120T machine → allow ALL 120T machines
    if normalized.intersection(MACHINE_GROUP_120T):
        part_machine_rows.append({
            "Child Part": child,
            "Possible Machines": ", ".join(MACHINE_GROUP_120T)
        })

part_machine_df = pd.DataFrame(part_machine_rows)

# =============================
# STEP 2: PART-WISE MACHINE CAPACITY (120T)
# =============================
capacity_rows = []

for _, row in part_machine_df.iterrows():
    child = row["Child Part"]
    ct = cycle_time_min.get(child)

    if not ct or ct <= 0:
        continue

    capacity = AVAILABLE_TIME_MIN / ct

    for machine in MACHINE_GROUP_120T:
        capacity_rows.append({
            "Child Part": child,
            "Machine": machine,
            "Capacity (units/day)": int(capacity)
        })

capacity_df = pd.DataFrame(capacity_rows)

# =============================
# WRITE TO EXCEL (2 SHEETS)
# =============================
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    part_machine_df.to_excel(writer, sheet_name="Part_Machine_Map", index=False)
    capacity_df.to_excel(writer, sheet_name="Part_Machine_Capacity", index=False)

print(f"✅ Excel file created: {output_file}")


In [ ]:
import pandas as pd

# =============================
# FILE PATH
# =============================
file_path = "input.xlsx"

# =============================
# LOAD DATA
# =============================
master = pd.read_excel(file_path, sheet_name="Master Data")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master")

# =============================
# CLEAN NUMERIC COLUMNS
# =============================
master["Net Required Qty"] = pd.to_numeric(
    master["Net Required Qty"], errors="coerce"
).fillna(0)

# =============================
# CYCLE TIME (SECONDS → MINUTES)
# =============================
# Priority:
# 1. Use Cycle Time from Master Data if present
# 2. Else use Part Production Master

if "Cycle Time" in master.columns:
    master["Cycle_Time_Min"] = pd.to_numeric(
        master["Cycle Time"], errors="coerce"
    ) / 60
else:
    ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce")
    cycle_time_map = (
        ppm.dropna(subset=["Material", "Machine"])
           .set_index("Material")["Machine"]
           .div(60)
           .to_dict()
    )
    master["Cycle_Time_Min"] = master["Child Part"].map(cycle_time_map)

# =============================
# BUILD PART LOAD TABLE
# =============================
parts_df = master[
    (master["Net Required Qty"] > 0) &
    (master["Cycle_Time_Min"] > 0)
].copy()

parts_df["Daily_Load_Min"] = (
    parts_df["Net Required Qty"] * parts_df["Cycle_Time_Min"]
)

parts_df = parts_df[["Child Part", "Daily_Load_Min"]]

# =============================
# CONSTANTS
# =============================
MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
DAILY_CAPACITY = 22 * 60      # 1320 minutes
CHANGEOVER_TIME = 40         # minutes

# =============================
# INITIAL MACHINE STATE
# =============================
machine_state = {
    m: {
        "load": 0.0,
        "parts_assigned": 0,
        "assignments": []
    }
    for m in MACHINES
}

# =============================
# SORT PARTS (HEAVIEST FIRST)
# =============================
parts_df = parts_df.sort_values("Daily_Load_Min", ascending=False)

# =============================
# LOAD BALANCING (CORRECT CHANGEOVER LOGIC)
# =============================
for _, row in parts_df.iterrows():

    part = row["Child Part"]
    part_load = row["Daily_Load_Min"]

    best_machine = None
    best_effective_load = float("inf")

    for m in MACHINES:
        m_state = machine_state[m]

        changeover_penalty = (
            CHANGEOVER_TIME if m_state["parts_assigned"] > 0 else 0
        )

        effective_load = (
            m_state["load"] + part_load + changeover_penalty
        )

        if effective_load < best_effective_load:
            best_effective_load = effective_load
            best_machine = m

    # =========================
    # ASSIGN PART
    # =========================
    if machine_state[best_machine]["parts_assigned"] > 0:
        machine_state[best_machine]["load"] += CHANGEOVER_TIME
        changeover = CHANGEOVER_TIME
    else:
        changeover = 0

    machine_state[best_machine]["load"] += part_load
    machine_state[best_machine]["parts_assigned"] += 1

    machine_state[best_machine]["assignments"].append({
        "Child Part": part,
        "Part Load (min)": round(part_load, 2),
        "Changeover (min)": changeover
    })

# =============================
# DISPLAY RESULTS
# =============================
print("\n========== MACHINE-WISE PLAN (120T) ==========\n")

for m in MACHINES:
    print(f"🔧 {m}")
    print(f"Total Load: {round(machine_state[m]['load'], 2)} min")
    if machine_state[m]["assignments"]:
        print(pd.DataFrame(machine_state[m]["assignments"]))
    else:
        print("No parts assigned")
    print("-" * 60)

# =============================
# SUMMARY
# =============================
summary_df = pd.DataFrame([
    {
        "Machine": m,
        "Total Load (min)": round(machine_state[m]["load"], 2),
        "Capacity (min)": DAILY_CAPACITY,
        "Overload (min)": round(machine_state[m]["load"] - DAILY_CAPACITY, 2),
        "No. of Parts": machine_state[m]["parts_assigned"]
    }
    for m in MACHINES
])

print("\n========== MACHINE LOAD SUMMARY ==========\n")
print(summary_df)


In [ ]:
import pandas as pd
import re
from collections import defaultdict

file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")

# =============================
# CLEAN NUMERIC COLUMNS
# =============================
for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =============================
# CYCLE TIME (SECONDS → MINUTES)
# =============================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 22 * 60      # 1320 minutes
CHANGEOVER_TIME = 40            # minutes

# =============================
# MACHINE NORMALIZER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: FILTER DAILY PLAN > 0
# =============================
valid = master[master["Daily Plan"] > 0].copy()

# =============================
# STEP 2: AGGREGATE AT CHILD PART LEVEL
# =============================
records = []

for child, grp in valid.groupby("Child Part", sort=False):

    demand_from_switches = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inventory = grp["Inventory_25"].iloc[0]

    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Required Qty": min_qty + demand_from_switches,
        "Inventory_25": inventory,
        "Vertical Machines": machines
    })

agg = pd.DataFrame(records)
agg["Net Required Qty"] = agg["Required Qty"] - agg["Inventory_25"]

# =============================
# PREPARE PART LOAD TABLE
# =============================
parts = []

for _, row in agg.iterrows():

    if row["Net Required Qty"] <= 0:
        continue

    ct = cycle_time_min.get(row["Child Part"])
    if not ct or ct <= 0:
        continue

    part_time = row["Net Required Qty"] * ct

    parts.append({
        "Child Part": row["Child Part"],
        "Time Required (min)": part_time,
        "Vertical Machines": row["Vertical Machines"]
    })

parts_df = pd.DataFrame(parts)

# Sort heavy parts first
parts_df = parts_df.sort_values("Time Required (min)", ascending=False)

# =============================
# MACHINE STATE (FOR BALANCING)
# =============================
machine_state = {
    m: {
        "load": 0.0,
        "parts_assigned": 0,
        "plan": []
    }
    for m in ALLOWED_MACHINES
}

# =============================
# STEP 3: BALANCED ASSIGNMENT WITH CHANGEOVER
# =============================
for _, row in parts_df.iterrows():

    child = row["Child Part"]
    part_time = row["Time Required (min)"]

    # Parse eligible machines
    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    eligible = []

    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in eligible:
            eligible.append(m)

    if not eligible:
        continue

    best_machine = None
    best_effective_load = float("inf")

    for m in eligible:
        state = machine_state[m]
        penalty = CHANGEOVER_TIME if state["parts_assigned"] > 0 else 0
        effective_load = state["load"] + part_time + penalty

        if effective_load < best_effective_load:
            best_effective_load = effective_load
            best_machine = m

    # Apply assignment
    if machine_state[best_machine]["parts_assigned"] > 0:
        machine_state[best_machine]["load"] += CHANGEOVER_TIME
        changeover = CHANGEOVER_TIME
    else:
        changeover = 0

    machine_state[best_machine]["load"] += part_time
    machine_state[best_machine]["parts_assigned"] += 1

    machine_state[best_machine]["plan"].append({
        "Child Part": child,
        "Time Used (min)": round(part_time, 2),
        "Changeover (min)": changeover
    })

# =============================
# DISPLAY RESULTS
# =============================
print("\n========== MACHINE-WISE BALANCED PLAN ==========\n")

for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if machine_state[m]["plan"]:
        display(pd.DataFrame(machine_state[m]["plan"]))
        print(f"Total Load: {round(machine_state[m]['load'], 2)} min")
    else:
        print("No allocation")
    print("-" * 60)

print("\n========== MACHINE LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_state[m]["load"], 2),
        "Capacity (min)": MACHINE_CAPACITY,
        "Overload (min)": round(machine_state[m]["load"] - MACHINE_CAPACITY, 2),
        "No. of Parts": machine_state[m]["parts_assigned"]
    }
    for m in ALLOWED_MACHINES
]))


In [ ]:
import pandas as pd
import re
from collections import defaultdict

# =============================
# FILE PATH
# =============================
file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

# =============================
# LOAD DATA
# =============================
master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")

# =============================
# CLEAN NUMERIC COLUMNS
# =============================
for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =============================
# CYCLE TIME (SECONDS → MINUTES)
# =============================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
DAILY_CAPACITY = 22 * 60     # 1320 minutes
CHANGEOVER_TIME = 40        # minutes

# =============================
# MACHINE NORMALIZER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: FILTER DAILY PLAN > 0
# =============================
valid = master[master["Daily Plan"] > 0].copy()

# =============================
# STEP 2: AGGREGATE & CALCULATE QTY TO MANUFACTURE
# =============================
records = []

for child, grp in valid.groupby("Child Part", sort=False):

    demand_from_switches = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inventory = grp["Inventory_25"].iloc[0]

    qty_to_manufacture = (demand_from_switches + min_qty) - inventory

    if qty_to_manufacture <= 0:
        continue

    ct = cycle_time_min.get(child)
    if not ct or ct <= 0:
        continue

    time_required = qty_to_manufacture * ct

    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Qty_to_Manufacture": qty_to_manufacture,
        "Time Required (min)": time_required,
        "Vertical Machines": machines
    })

parts_df = pd.DataFrame(records)

# =============================
# PRIORITIZE HIGH-RISK PARTS
# =============================
parts_df = parts_df.sort_values("Time Required (min)", ascending=False)

# =============================
# MACHINE STATE
# =============================
machine_state = {
    m: {
        "used_time": 0.0,
        "parts": []
    }
    for m in MACHINES
}

# =============================
# STEP 3: GROUND-LEVEL PLANNING
# =============================
final_plan = []

for _, row in parts_df.iterrows():

    part = row["Child Part"]
    remaining_time = row["Time Required (min)"]

    # Eligible machines for this part
    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    eligible = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in MACHINES and m not in eligible:
            eligible.append(m)

    if not eligible:
        continue

    # ---- TRY SINGLE MACHINE FIRST (MIN CHANGEOVER) ----
    placed = False
    for m in eligible:
        available = DAILY_CAPACITY - machine_state[m]["used_time"]
        if available >= remaining_time:
            changeover = 0 if len(machine_state[m]["parts"]) == 0 else CHANGEOVER_TIME
            machine_state[m]["used_time"] += remaining_time + changeover
            machine_state[m]["parts"].append(part)

            final_plan.append({
                "Child Part": part,
                "Machine": m,
                "Time Used (min)": round(remaining_time, 2),
                "Changeover (min)": changeover
            })

            placed = True
            break

    if placed:
        continue

    # ---- SPLIT ACROSS MINIMUM MACHINES ----
    eligible_sorted = sorted(
        eligible,
        key=lambda x: DAILY_CAPACITY - machine_state[x]["used_time"],
        reverse=True
    )

    for m in eligible_sorted:
        if remaining_time <= 0:
            break

        available = DAILY_CAPACITY - machine_state[m]["used_time"]
        if available <= 0:
            continue

        alloc = min(available, remaining_time)
        changeover = 0 if len(machine_state[m]["parts"]) == 0 else CHANGEOVER_TIME

        machine_state[m]["used_time"] += alloc + changeover
        machine_state[m]["parts"].append(part)

        final_plan.append({
            "Child Part": part,
            "Machine": m,
            "Time Used (min)": round(alloc, 2),
            "Changeover (min)": changeover
        })

        remaining_time -= alloc

# =============================
# OUTPUT
# =============================
plan_df = pd.DataFrame(final_plan)

print("\n========== FINAL PRODUCTION PLAN ==========\n")
display(plan_df)

print("\n========== MACHINE LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_state[m]["used_time"], 2),
        "Capacity (min)": DAILY_CAPACITY,
        "Overload (min)": round(machine_state[m]["used_time"] - DAILY_CAPACITY, 2),
        "No. of Parts": len(machine_state[m]["parts"])
    }
    for m in MACHINES
]))


In [ ]:
import pandas as pd
import re
from collections import defaultdict
import math

file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")

# =============================
# CLEAN NUMERIC COLUMNS
# =============================
for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =============================
# CYCLE TIME (SECONDS → MINUTES)
# =============================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
DAILY_CAPACITY = 22 * 60        # 1320 min
CHANGEOVER_TIME = 40            # min

LOW_DEMAND_THRESHOLD = 100      # pcs/day
BATCH_RUN_TIME_MIN = 180        # cycle-time based batching (3 hrs)

# =============================
# MACHINE NORMALIZER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: DAILY DEMAND PER PART
# =============================
valid = master[master["Daily Plan"] > 0].copy()

records = []

for child, grp in valid.groupby("Child Part", sort=False):

    daily_demand = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inventory = grp["Inventory_25"].iloc[0]

    ct = cycle_time_min.get(child)
    if not ct or ct <= 0:
        continue

    inventory_gap = max(0, min_qty - inventory)

    # -----------------------------
    # DECIDE BATCH QTY (KEY CHANGE)
    # -----------------------------
    if daily_demand >= LOW_DEMAND_THRESHOLD:
        batch_qty = max(daily_demand, inventory_gap)
    else:
        if inventory >= min_qty:
            continue  # skip low-demand, healthy stock parts
        batch_qty = math.ceil(BATCH_RUN_TIME_MIN / ct)

    time_required = batch_qty * ct
    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Batch Qty": batch_qty,
        "Time Required (min)": time_required,
        "Vertical Machines": machines
    })

parts_df = pd.DataFrame(records)

# =============================
# PRIORITIZE HEAVY PARTS
# =============================
parts_df = parts_df.sort_values("Time Required (min)", ascending=False)

# =============================
# MACHINE STATE
# =============================
machine_state = {
    m: {"used": 0.0, "parts": []}
    for m in ALLOWED_MACHINES
}

final_plan = []

# =============================
# STEP 2: SCHEDULING WITH MIN SPLIT
# =============================
for _, row in parts_df.iterrows():

    part = row["Child Part"]
    remaining = row["Time Required (min)"]

    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    eligible = []

    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in eligible:
            eligible.append(m)

    if not eligible:
        continue

    # ---- TRY SINGLE MACHINE FIRST ----
    placed = False
    for m in eligible:
        available = DAILY_CAPACITY - machine_state[m]["used"]
        if available >= remaining:
            changeover = 0 if len(machine_state[m]["parts"]) == 0 else CHANGEOVER_TIME
            machine_state[m]["used"] += remaining + changeover
            machine_state[m]["parts"].append(part)

            final_plan.append({
                "Child Part": part,
                "Machine": m,
                "Time Used (min)": round(remaining, 2),
                "Changeover (min)": changeover
            })

            placed = True
            break

    if placed:
        continue

    # ---- SPLIT ACROSS MIN MACHINES ----
    eligible_sorted = sorted(
        eligible,
        key=lambda x: DAILY_CAPACITY - machine_state[x]["used"],
        reverse=True
    )

    for m in eligible_sorted:
        if remaining <= 0:
            break

        available = DAILY_CAPACITY - machine_state[m]["used"]
        if available <= 0:
            continue

        alloc = min(available, remaining)
        changeover = 0 if len(machine_state[m]["parts"]) == 0 else CHANGEOVER_TIME

        machine_state[m]["used"] += alloc + changeover
        machine_state[m]["parts"].append(part)

        final_plan.append({
            "Child Part": part,
            "Machine": m,
            "Time Used (min)": round(alloc, 2),
            "Changeover (min)": changeover
        })

        remaining -= alloc

# =============================
# OUTPUT
# =============================
print("\n========== FINAL PRODUCTION PLAN ==========\n")
display(pd.DataFrame(final_plan))

print("\n========== MACHINE LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_state[m]["used"], 2),
        "Capacity (min)": DAILY_CAPACITY,
        "Overload (min)": round(machine_state[m]["used"] - DAILY_CAPACITY, 2),
        "No. of Parts": len(machine_state[m]["parts"])
    }
    for m in ALLOWED_MACHINES
]))


In [ ]:
# ==========================================================
# PASS 1: RAW SCHEDULE (NO BALANCING, FIRST ELIGIBLE MACHINE)
# ==========================================================

raw_machine_state = {
    m: {"used": 0.0, "plan": []}
    for m in ALLOWED_MACHINES
}

raw_plan = []

for _, row in parts_df.iterrows():

    part = row["Child Part"]
    time_required = row["Time Required (min)"]

    # eligible machines (same parsing)
    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    eligible = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in eligible:
            eligible.append(m)

    if not eligible:
        continue

    # 🔴 NO THINKING: FIRST MACHINE ONLY
    m = eligible[0]

    changeover = 0 if len(raw_machine_state[m]["plan"]) == 0 else CHANGEOVER_TIME
    raw_machine_state[m]["used"] += time_required + changeover

    raw_machine_state[m]["plan"].append({
        "Child Part": part,
        "Time Used (min)": round(time_required, 2),
        "Changeover (min)": changeover
    })

    raw_plan.append({
        "Child Part": part,
        "Machine": m,
        "Time Used (min)": round(time_required, 2),
        "Changeover (min)": changeover
    })

print("\n========== RAW MACHINE-WISE SCHEDULE (NO BALANCING) ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if raw_machine_state[m]["plan"]:
        display(pd.DataFrame(raw_machine_state[m]["plan"]))
        print(f"Total Load: {round(raw_machine_state[m]['used'], 2)} min")
    else:
        print("No allocation")
    print("-" * 60)

print("\n========== RAW LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(raw_machine_state[m]["used"], 2),
        "Capacity (min)": DAILY_CAPACITY,
        "Overload (min)": round(raw_machine_state[m]["used"] - DAILY_CAPACITY, 2)
    }
    for m in ALLOWED_MACHINES
]))


In [ ]:
# ==========================================================
# PASS 1: RAW SCHEDULE (NO BALANCING, FIRST ELIGIBLE MACHINE)
# ==========================================================

raw_machine_state = {
    m: {"used": 0.0, "plan": []}
    for m in ALLOWED_MACHINES
}

raw_plan = []

for _, row in parts_df.iterrows():

    part = row["Child Part"]
    time_required = row["Time Required (min)"]

    # eligible machines (same parsing)
    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    eligible = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in eligible:
            eligible.append(m)

    if not eligible:
        continue

    # 🔴 NO THINKING: FIRST MACHINE ONLY
    m = eligible[0]

    changeover = 0 if len(raw_machine_state[m]["plan"]) == 0 else CHANGEOVER_TIME
    raw_machine_state[m]["used"] += time_required + changeover

    raw_machine_state[m]["plan"].append({
        "Child Part": part,
        "Time Used (min)": round(time_required, 2),
        "Changeover (min)": changeover
    })

    raw_plan.append({
        "Child Part": part,
        "Machine": m,
        "Time Used (min)": round(time_required, 2),
        "Changeover (min)": changeover
    })

print("\n========== RAW MACHINE-WISE SCHEDULE (NO BALANCING) ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if raw_machine_state[m]["plan"]:
        display(pd.DataFrame(raw_machine_state[m]["plan"]))
        print(f"Total Load: {round(raw_machine_state[m]['used'], 2)} min")
    else:
        print("No allocation")
    print("-" * 60)

print("\n========== RAW LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(raw_machine_state[m]["used"], 2),
        "Capacity (min)": DAILY_CAPACITY,
        "Overload (min)": round(raw_machine_state[m]["used"] - DAILY_CAPACITY, 2)
    }
    for m in ALLOWED_MACHINES
]))


In [ ]:
import pandas as pd
import re
import math

# =========================================================
# FILE PATH
# =========================================================
file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

# =========================================================
# LOAD DATA
# =========================================================
master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")

# =========================================================
# CLEAN NUMERIC COLUMNS
# =========================================================
for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =========================================================
# CYCLE TIME (SECONDS → MINUTES)
# =========================================================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =========================================================
# CONSTANTS
# =========================================================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
DAILY_CAPACITY = 22 * 60          # 1320 min
CHANGEOVER_TIME = 40              # min

LOW_DEMAND_THRESHOLD = 100        # pcs/day
BATCH_RUN_TIME_MIN = 180          # 3 hrs batching

# =========================================================
# MACHINE NORMALIZER
# =========================================================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =========================================================
# STEP 1: DAILY DEMAND + BATCH DECISION
# =========================================================
valid = master[master["Daily Plan"] > 0].copy()
records = []

for child, grp in valid.groupby("Child Part", sort=False):

    daily_demand = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inventory = grp["Inventory_25"].iloc[0]

    ct = cycle_time_min.get(child)
    if not ct or ct <= 0:
        continue

    inventory_gap = max(0, min_qty - inventory)

    # ---------- BATCH LOGIC ----------
    if daily_demand >= LOW_DEMAND_THRESHOLD:
        batch_qty = max(daily_demand, inventory_gap)
    else:
        if inventory >= min_qty:
            continue
        batch_qty = math.ceil(BATCH_RUN_TIME_MIN / ct)

    time_required = batch_qty * ct
    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Batch Qty": batch_qty,
        "Time Required (min)": time_required,
        "Vertical Machines": machines
    })

parts_df = pd.DataFrame(records)
parts_df = parts_df.sort_values("Time Required (min)", ascending=False)

# =========================================================
# PASS 1: RAW SCHEDULE (NO BALANCING)
# =========================================================
raw_state = {m: {"used": 0.0, "plan": []} for m in ALLOWED_MACHINES}

for _, row in parts_df.iterrows():
    part = row["Child Part"]
    time_req = row["Time Required (min)"]

    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    eligible = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES:
            eligible.append(m)
            break

    if not eligible:
        continue

    m = eligible[0]
    changeover = 0 if len(raw_state[m]["plan"]) == 0 else CHANGEOVER_TIME
    raw_state[m]["used"] += time_req + changeover

    raw_state[m]["plan"].append({
        "Child Part": part,
        "Time Used (min)": round(time_req, 2),
        "Changeover (min)": changeover
    })

print("\n================ RAW MACHINE-WISE SCHEDULE ================\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if raw_state[m]["plan"]:
        display(pd.DataFrame(raw_state[m]["plan"]))
        print(f"Total Load: {round(raw_state[m]['used'],2)} min")
    else:
        print("No allocation")
    print("-"*60)

print("\n================ RAW LOAD SUMMARY ================\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(raw_state[m]["used"], 2),
        "Capacity (min)": DAILY_CAPACITY,
        "Overload (min)": round(raw_state[m]["used"] - DAILY_CAPACITY, 2)
    }
    for m in ALLOWED_MACHINES
]))

# =========================================================
# PASS 2: BALANCED SCHEDULE (PLANNER LOGIC)
# =========================================================
bal_state = {m: {"used": 0.0, "plan": []} for m in ALLOWED_MACHINES}

for _, row in parts_df.iterrows():

    part = row["Child Part"]
    remaining = row["Time Required (min)"]

    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    eligible = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in eligible:
            eligible.append(m)

    if not eligible:
        continue

    # ---- Try single machine first ----
    placed = False
    for m in eligible:
        available = DAILY_CAPACITY - bal_state[m]["used"]
        if available >= remaining:
            changeover = 0 if len(bal_state[m]["plan"]) == 0 else CHANGEOVER_TIME
            bal_state[m]["used"] += remaining + changeover
            bal_state[m]["plan"].append({
                "Child Part": part,
                "Time Used (min)": round(remaining,2),
                "Changeover (min)": changeover
            })
            placed = True
            break

    if placed:
        continue

    # ---- Split across minimum machines ----
    eligible_sorted = sorted(
        eligible,
        key=lambda x: DAILY_CAPACITY - bal_state[x]["used"],
        reverse=True
    )

    for m in eligible_sorted:
        if remaining <= 0:
            break

        available = DAILY_CAPACITY - bal_state[m]["used"]
        if available <= 0:
            continue

        alloc = min(available, remaining)
        changeover = 0 if len(bal_state[m]["plan"]) == 0 else CHANGEOVER_TIME
        bal_state[m]["used"] += alloc + changeover

        bal_state[m]["plan"].append({
            "Child Part": part,
            "Time Used (min)": round(alloc,2),
            "Changeover (min)": changeover
        })

        remaining -= alloc

print("\n================ BALANCED MACHINE-WISE SCHEDULE ================\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if bal_state[m]["plan"]:
        display(pd.DataFrame(bal_state[m]["plan"]))
        print(f"Total Load: {round(bal_state[m]['used'],2)} min")
    else:
        print("No allocation")
    print("-"*60)

print("\n================ BALANCED LOAD SUMMARY ================\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(bal_state[m]["used"], 2),
        "Capacity (min)": DAILY_CAPACITY,
        "Overload (min)": round(bal_state[m]["used"] - DAILY_CAPACITY, 2)
    }
    for m in ALLOWED_MACHINES
]))


In [ ]:
import pandas as pd
import re
import math

# =========================================================
# FILE PATH
# =========================================================
file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

# =========================================================
# LOAD DATA
# =========================================================
master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")

# =========================================================
# CLEAN NUMERIC COLUMNS
# =========================================================
for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =========================================================
# CYCLE TIME (SECONDS → MINUTES)
# =========================================================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =========================================================
# CONSTANTS
# =========================================================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
DAILY_CAPACITY = 22 * 60        # 1320 minutes
CHANGEOVER_TIME = 40            # minutes

LOW_DEMAND_THRESHOLD = 100      # pcs/day
BATCH_RUN_TIME_MIN = 180        # 3 hours batching

# =========================================================
# MACHINE NORMALIZER
# =========================================================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =========================================================
# STEP 1: PART-WISE BATCH DECISION
# =========================================================
valid = master[master["Daily Plan"] > 0].copy()
records = []

for child, grp in valid.groupby("Child Part", sort=False):

    daily_demand = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inventory = grp["Inventory_25"].iloc[0]

    ct = cycle_time_min.get(child)
    if not ct or ct <= 0:
        continue

    inventory_gap = max(0, min_qty - inventory)

    if daily_demand >= LOW_DEMAND_THRESHOLD:
        batch_qty = max(daily_demand, inventory_gap)
    else:
        if inventory >= min_qty:
            continue
        batch_qty = math.ceil(BATCH_RUN_TIME_MIN / ct)

    time_required = batch_qty * ct
    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Batch Qty": batch_qty,
        "Cycle Time (min)": ct,
        "Time Required (min)": time_required,
        "Vertical Machines": machines
    })

parts_df = pd.DataFrame(records).sort_values("Time Required (min)", ascending=False)

# =========================================================
# DISPLAY PART-WISE PLANNED QTY
# =========================================================
print("\n========== PART-WISE PLANNED PRODUCTION ==========\n")
display(parts_df[["Child Part", "Batch Qty", "Time Required (min)"]])

# =========================================================
# PASS 1: RAW SCHEDULE (NO BALANCING)
# =========================================================
raw_state = {m: {"used": 0.0, "plan": []} for m in ALLOWED_MACHINES}
raw_plan = []

for _, row in parts_df.iterrows():

    part = row["Child Part"]
    ct = row["Cycle Time (min)"]
    time_req = row["Time Required (min)"]
    qty = round(time_req / ct)

    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    machine = None
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES:
            machine = m
            break

    if not machine:
        continue

    changeover = 0 if not raw_state[machine]["plan"] else CHANGEOVER_TIME
    raw_state[machine]["used"] += time_req + changeover

    raw_state[machine]["plan"].append({
        "Child Part": part,
        "Quantity": qty,
        "Time Used (min)": round(time_req, 2),
        "Changeover (min)": changeover
    })

    raw_plan.append({
        "Child Part": part,
        "Machine": machine,
        "Quantity": qty
    })

print("\n========== RAW MACHINE-WISE SCHEDULE ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    display(pd.DataFrame(raw_state[m]["plan"]))
    print(f"Total Load: {round(raw_state[m]['used'],2)} min")
    print("-"*60)

# =========================================================
# PASS 2: BALANCED SCHEDULE
# =========================================================
bal_state = {m: {"used": 0.0, "plan": []} for m in ALLOWED_MACHINES}
final_plan = []

for _, row in parts_df.iterrows():

    part = row["Child Part"]
    ct = row["Cycle Time (min)"]
    remaining = row["Time Required (min)"]

    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    eligible = [normalize_machine(t) for t in tokens if normalize_machine(t) in ALLOWED_MACHINES]

    if not eligible:
        continue

    # Try single machine
    placed = False
    for m in eligible:
        available = DAILY_CAPACITY - bal_state[m]["used"]
        if available >= remaining:
            changeover = 0 if not bal_state[m]["plan"] else CHANGEOVER_TIME
            bal_state[m]["used"] += remaining + changeover
            qty = round(remaining / ct)

            bal_state[m]["plan"].append({
                "Child Part": part,
                "Quantity": qty,
                "Time Used (min)": round(remaining,2),
                "Changeover (min)": changeover
            })

            final_plan.append({
                "Child Part": part,
                "Machine": m,
                "Quantity": qty
            })

            placed = True
            break

    if placed:
        continue

    # Split across minimum machines
    eligible_sorted = sorted(
        eligible,
        key=lambda x: DAILY_CAPACITY - bal_state[x]["used"],
        reverse=True
    )

    for m in eligible_sorted:
        if remaining <= 0:
            break

        available = DAILY_CAPACITY - bal_state[m]["used"]
        if available <= 0:
            continue

        alloc = min(available, remaining)
        changeover = 0 if not bal_state[m]["plan"] else CHANGEOVER_TIME

        bal_state[m]["used"] += alloc + changeover
        qty = round(alloc / ct)

        bal_state[m]["plan"].append({
            "Child Part": part,
            "Quantity": qty,
            "Time Used (min)": round(alloc,2),
            "Changeover (min)": changeover
        })

        final_plan.append({
            "Child Part": part,
            "Machine": m,
            "Quantity": qty
        })

        remaining -= alloc

print("\n========== BALANCED MACHINE-WISE SCHEDULE ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    display(pd.DataFrame(bal_state[m]["plan"]))
    print(f"Total Load: {round(bal_state[m]['used'],2)} min")
    print("-"*60)

# =========================================================
# PART-WISE QTY VALIDATION
# =========================================================
final_df = pd.DataFrame(final_plan)

qty_check = (
    final_df
    .groupby("Child Part", as_index=False)["Quantity"]
    .sum()
    .merge(parts_df[["Child Part", "Batch Qty"]], on="Child Part", how="left")
)

qty_check["Difference"] = qty_check["Batch Qty"] - qty_check["Quantity"]

print("\n========== PART-WISE PLANNED vs ALLOCATED QTY ==========\n")
display(qty_check)


In [ ]:
import pandas as pd
import re
import math

# =========================================================
# FILE PATH
# =========================================================
file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

# =========================================================
# LOAD DATA
# =========================================================
master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")

# =========================================================
# CLEAN NUMERIC COLUMNS
# =========================================================
for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =========================================================
# CYCLE TIME (SECONDS → MINUTES)
# =========================================================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =========================================================
# CONSTANTS
# =========================================================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
DAILY_CAPACITY = 22 * 60      # 1320 minutes
CHANGEOVER_TIME = 40          # minutes

BATCH_RUN_TIME_MIN = 180      # cycle-time based batching (execution aid only)

# =========================================================
# MACHINE NORMALIZER
# =========================================================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =========================================================
# STEP 1: COMPUTE NET REQUIRED QTY (CORRECT LOGIC)
# =========================================================
valid = master[master["Daily Plan"] > 0].copy()
records = []

for child, grp in valid.groupby("Child Part", sort=False):

    daily_demand = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inventory = grp["Inventory_25"].iloc[0]

    net_required_qty = daily_demand + min_qty - inventory

    if net_required_qty <= 0:
        continue  # inventory healthy

    ct = cycle_time_min.get(child)
    if not ct or ct <= 0:
        continue

    # -------- EXECUTION BATCH (DOES NOT REDUCE REQUIREMENT) --------
    # If net qty is very small, still run at least a cycle-time-justified batch
    min_batch_qty = math.ceil(BATCH_RUN_TIME_MIN / ct)
    batch_qty = max(net_required_qty, min_batch_qty)

    time_required = batch_qty * ct
    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Net Required Qty": net_required_qty,
        "Planned Qty": batch_qty,
        "Cycle Time (min)": ct,
        "Time Required (min)": time_required,
        "Vertical Machines": machines
    })

parts_df = pd.DataFrame(records).sort_values("Time Required (min)", ascending=False)

# =========================================================
# DISPLAY PART-WISE PLAN
# =========================================================
print("\n========== PART-WISE PRODUCTION PLAN ==========\n")
display(parts_df[[
    "Child Part",
    "Net Required Qty",
    "Planned Qty",
    "Time Required (min)"
]])

# =========================================================
# PASS 1: RAW SCHEDULE (NO BALANCING)
# =========================================================
raw_state = {m: {"used": 0.0, "plan": []} for m in ALLOWED_MACHINES}

for _, row in parts_df.iterrows():

    part = row["Child Part"]
    ct = row["Cycle Time (min)"]
    time_req = row["Time Required (min)"]
    qty = round(time_req / ct)

    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    machine = None
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES:
            machine = m
            break

    if not machine:
        continue

    changeover = 0 if not raw_state[machine]["plan"] else CHANGEOVER_TIME
    raw_state[machine]["used"] += time_req + changeover

    raw_state[machine]["plan"].append({
        "Child Part": part,
        "Quantity": qty,
        "Time Used (min)": round(time_req, 2),
        "Changeover (min)": changeover
    })

print("\n========== RAW MACHINE-WISE SCHEDULE ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    display(pd.DataFrame(raw_state[m]["plan"]))
    print(f"Total Load: {round(raw_state[m]['used'],2)} min")
    print("-"*60)

# =========================================================
# PASS 2: BALANCED SCHEDULE (MIN CHANGEOVER)
# =========================================================
bal_state = {m: {"used": 0.0, "plan": []} for m in ALLOWED_MACHINES}
final_plan = []

for _, row in parts_df.iterrows():

    part = row["Child Part"]
    ct = row["Cycle Time (min)"]
    remaining = row["Time Required (min)"]

    tokens = re.split(r"[,\|/\\\n]+", str(row["Vertical Machines"]))
    eligible = [normalize_machine(t) for t in tokens if normalize_machine(t) in ALLOWED_MACHINES]

    if not eligible:
        continue

    # ---- Try single machine first ----
    placed = False
    for m in eligible:
        available = DAILY_CAPACITY - bal_state[m]["used"]
        if available >= remaining:
            changeover = 0 if not bal_state[m]["plan"] else CHANGEOVER_TIME
            bal_state[m]["used"] += remaining + changeover
            qty = round(remaining / ct)

            bal_state[m]["plan"].append({
                "Child Part": part,
                "Quantity": qty,
                "Time Used (min)": round(remaining,2),
                "Changeover (min)": changeover
            })

            final_plan.append({
                "Child Part": part,
                "Machine": m,
                "Quantity": qty
            })

            placed = True
            break

    if placed:
        continue

    # ---- Split across minimum machines ----
    eligible_sorted = sorted(
        eligible,
        key=lambda x: DAILY_CAPACITY - bal_state[x]["used"],
        reverse=True
    )

    for m in eligible_sorted:
        if remaining <= 0:
            break

        available = DAILY_CAPACITY - bal_state[m]["used"]
        if available <= 0:
            continue

        alloc = min(available, remaining)
        changeover = 0 if not bal_state[m]["plan"] else CHANGEOVER_TIME

        bal_state[m]["used"] += alloc + changeover
        qty = round(alloc / ct)

        bal_state[m]["plan"].append({
            "Child Part": part,
            "Quantity": qty,
            "Time Used (min)": round(alloc,2),
            "Changeover (min)": changeover
        })

        final_plan.append({
            "Child Part": part,
            "Machine": m,
            "Quantity": qty
        })

        remaining -= alloc

print("\n========== BALANCED MACHINE-WISE SCHEDULE ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    display(pd.DataFrame(bal_state[m]["plan"]))
    print(f"Total Load: {round(bal_state[m]['used'],2)} min")
    print("-"*60)

# =========================================================
# PART-WISE QTY VALIDATION
# =========================================================
final_df = pd.DataFrame(final_plan)

qty_check = (
    final_df
    .groupby("Child Part", as_index=False)["Quantity"]
    .sum()
    .merge(parts_df[["Child Part", "Planned Qty"]], on="Child Part", how="left")
)

qty_check["Difference"] = qty_check["Planned Qty"] - qty_check["Quantity"]

print("\n========== PART-WISE PLANNED vs ALLOCATED QTY ==========\n")
display(qty_check)


In [ ]:
import pandas as pd
import re
from collections import defaultdict

file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")

for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =============================
# CYCLE TIME (SECONDS → MINUTES)
# =============================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 22 * 60  # 1320 minutes (1 day)
CHANGEOVER_MIN = 40.0

# =============================
# MACHINE NORMALIZER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: FILTER DAILY PLAN > 0
# =============================
valid = master[master["Daily Plan"] > 0].copy()

# =============================
# STEP 2: AGGREGATE AT CHILD PART LEVEL
# =============================
records = []

for child, grp in valid.groupby("Child Part", sort=False):

    demand_from_switches = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inventory = grp["Inventory_25"].iloc[0]

    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Required Qty": min_qty + demand_from_switches,
        "Inventory_25": inventory,
        "Vertical Machines": machines
    })

agg = pd.DataFrame(records)
agg["Net Required Qty"] = agg["Required Qty"] - agg["Inventory_25"]

# =============================
# TRACKING
# =============================
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)

# =============================
# STEP 3: ASSIGN WITH LOAD BALANCING AND CHANGEOVER
# =============================
for _, row in agg.iterrows():

    child = row["Child Part"]
    net_qty = row["Net Required Qty"]

    if net_qty <= 0:
        continue

    ct = cycle_time_min.get(child)
    if not ct or ct <= 0:
        continue

    total_time = net_qty * ct

    # Parse machines in given order
    raw = str(row["Vertical Machines"])
    tokens = re.split(r"[,\|/\\\n]+", raw)

    eligible = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in eligible:
            eligible.append(m)

    if not eligible:
        continue

    # Calculate "effective load" if we assign this part to each machine
    best_machine = None
    best_effective_load = float('inf')

    for m in eligible:
        current_load = machine_load[m]
        # If this machine already has SOME parts planned → add changeover
        num_existing = len(machine_plan[m])
        changeover_penalty = CHANGEOVER_MIN if num_existing > 0 else 0.0
        
        effective = current_load + total_time + changeover_penalty
        if effective < best_effective_load:
            best_effective_load = effective
            best_machine = m

    if best_machine is None:
        continue

    # Assign to best
    changeover_added = CHANGEOVER_MIN if len(machine_plan[best_machine]) > 0 else 0.0
    
    machine_load[best_machine] += total_time + changeover_added
    
    machine_plan[best_machine].append({
        "Child Part": child,
        "Quantity": round(net_qty, 2),
        "Time Used (production)": round(total_time, 2),
        "Changeover (min)": round(changeover_added, 2),
        "Total Time Added": round(total_time + changeover_added, 2)
    })

# =============================
# DISPLAY RESULTS
# =============================
print("\n========== MACHINE-WISE PLAN (1 DAY, WITH LOAD BALANCING AND CHANGEOVER) ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No allocation")
    print("-" * 60)

print("\n========== MACHINE LOAD SUMMARY (OVERLOAD VISIBLE) ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Capacity (min)": MACHINE_CAPACITY,
        "Overload (min)": round(machine_load[m] - MACHINE_CAPACITY, 2)
    }
    for m in ALLOWED_MACHINES
]))

In [ ]:
import pandas as pd
import re
from collections import defaultdict

file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")

for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# Cycle time dict (seconds → minutes)
cycle_time_min = {row["Material"]: row["Machine"] / 60 for _, row in ppm.iterrows()}

# Constants
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 22 * 60          # 1320 min
CHANGEOVER_MIN   = 40.0

def normalize_machine(m):
    if not m or pd.isna(m): return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# ───────────────────────────────────────────────
# Prepare data
# ───────────────────────────────────────────────
valid = master[master["Daily Plan"] > 0].copy()

records = []
for child, grp in valid.groupby("Child Part", sort=False):
    daily_demand = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty      = grp["Minimum Quantity"].iloc[0]
    inventory    = grp["Inventory_25"].iloc[0]
    machines_str = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Daily Demand": daily_demand,
        "Buffer Needed": min_qty,
        "Inventory": inventory,
        "Net Required": daily_demand + min_qty - inventory,
        "Vertical Machines": machines_str
    })

df_parts = pd.DataFrame(records)
df_parts["Net Required"] = df_parts["Net Required"].clip(lower=0)

# Tracking
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)  # list of assignment dicts

# ───────────────────────────────────────────────
# Helper: get eligible machines
# ───────────────────────────────────────────────
def get_eligible_machines(machines_str):
    raw = str(machines_str)
    tokens = re.split(r"[,\|/\\\n]+", raw)
    eligible = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in eligible:
            eligible.append(m)
    return eligible

# ───────────────────────────────────────────────
# Phase 1: Assign full Daily Demand (priority)
# ───────────────────────────────────────────────
daily_parts = df_parts[df_parts["Daily Demand"] > 0].copy()
daily_parts["Prod Time"] = daily_parts["Daily Demand"] * daily_parts["Child Part"].map(cycle_time_min).fillna(0)
daily_parts = daily_parts[daily_parts["Prod Time"] > 0]
daily_parts = daily_parts.sort_values("Prod Time", ascending=False)  # largest first

for _, row in daily_parts.iterrows():
    child = row["Child Part"]
    qty = row["Daily Demand"]
    ct = cycle_time_min.get(child, 0)
    if ct <= 0: continue
    total_time = qty * ct

    eligible = get_eligible_machines(row["Vertical Machines"])
    if not eligible: continue

    # Pick machine with least current load (after penalty)
    best_m = None
    best_score = float('inf')

    for m in eligible:
        penalty = CHANGEOVER_MIN if len(machine_plan[m]) > 0 else 0
        score = machine_load[m] + total_time + penalty
        if score < best_score:
            best_score = score
            best_m = m

    if best_m is None: continue

    # Check if fits
    remaining = MACHINE_CAPACITY - machine_load[best_m]
    penalty = CHANGEOVER_MIN if len(machine_plan[best_m]) > 0 else 0
    if remaining >= total_time + penalty:
        changeover = penalty
        assigned_qty = qty
        prod_time = total_time
    else:
        # Cannot fit full daily → assign partial (foreman tries hard but capacity rules)
        avail_after_change = remaining - penalty
        if avail_after_change <= 0: continue
        assigned_qty = avail_after_change / ct
        prod_time = assigned_qty * ct
        changeover = penalty

    machine_load[best_m] += prod_time + changeover
    machine_plan[best_m].append({
        "Child Part": child,
        "Type": "Daily",
        "Quantity": round(assigned_qty, 2),
        "Prod Time": round(prod_time, 2),
        "Changeover": round(changeover, 2),
        "Total Added": round(prod_time + changeover, 2)
    })

# ───────────────────────────────────────────────
# Phase 2: Fill remaining capacity with Buffer
# ───────────────────────────────────────────────
buffer_parts = df_parts[df_parts["Buffer Needed"] > 0].copy()
buffer_parts["Possible Buffer Time"] = buffer_parts["Buffer Needed"] * buffer_parts["Child Part"].map(cycle_time_min).fillna(0)
buffer_parts = buffer_parts.sort_values("Possible Buffer Time", ascending=False)

for _, row in buffer_parts.iterrows():
    child = row["Child Part"]
    ct = cycle_time_min.get(child, 0)
    if ct <= 0: continue

    eligible = get_eligible_machines(row["Vertical Machines"])
    if not eligible: continue

    remaining_buffer = row["Buffer Needed"]   # we try to add as much as possible

    while remaining_buffer > 0:
        # Find machine with most remaining capacity
        best_m = None
        best_remaining = -1

        for m in eligible:
            rem = MACHINE_CAPACITY - machine_load[m]
            penalty = CHANGEOVER_MIN if len(machine_plan[m]) > 0 else 0
            avail = rem - penalty
            if avail > best_remaining:
                best_remaining = avail
                best_m = m

        if best_m is None or best_remaining <= 0:
            break  # no more space

        qty_can_make = best_remaining / ct
        assign_qty = min(remaining_buffer, qty_can_make)
        prod_time = assign_qty * ct
        changeover = CHANGEOVER_MIN if len(machine_plan[best_m]) > 0 else 0

        machine_load[best_m] += prod_time + changeover
        machine_plan[best_m].append({
            "Child Part": child,
            "Type": "Buffer",
            "Quantity": round(assign_qty, 2),
            "Prod Time": round(prod_time, 2),
            "Changeover": round(changeover, 2),
            "Total Added": round(prod_time + changeover, 2)
        })

        remaining_buffer -= assign_qty

# ───────────────────────────────────────────────
# Output
# ───────────────────────────────────────────────
print("\n===== MACHINE-WISE PRODUCTION PLAN (No Overload) =====\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if machine_plan[m]:
        df_plan = pd.DataFrame(machine_plan[m])
        print(df_plan)
    else:
        print("No allocation")
    util = round(100 * machine_load[m] / MACHINE_CAPACITY, 1)
    print(f"→ Used: {round(machine_load[m],1)} min / {MACHINE_CAPACITY} min ({util}%)\n")

print("===== SUMMARY =====\n")
summary = pd.DataFrame([
    {"Machine": m, "Used min": round(machine_load[m],1), 
     "Util %": round(100 * machine_load[m]/MACHINE_CAPACITY,1)}
    for m in ALLOWED_MACHINES
])
print(summary)

In [ ]:
import pandas as pd
import re
from collections import defaultdict

# =============================
# FILE
# =============================
file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")

# =============================
# CLEAN NUMERIC COLUMNS
# =============================
for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =============================
# CYCLE TIME (SECONDS → MINUTES)
# =============================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 22 * 60          # 1320 min/day
CHANGEOVER_TIME = 40               # min per machine-part
SAFE_BUFFER_DAYS = 1               # foreman rule

# =============================
# MACHINE NORMALIZER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: FILTER DAILY PLAN > 0
# =============================
valid = master[master["Daily Plan"] > 0].copy()

# =============================
# STEP 2: AGGREGATE AT CHILD PART
# =============================
records = []

for child, grp in valid.groupby("Child Part", sort=False):

    daily_demand = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inventory = grp["Inventory_25"].iloc[0]
    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Daily Demand": daily_demand,
        "Required Qty": daily_demand + min_qty,
        "Inventory_25": inventory,
        "Vertical Machines": machines
    })

agg = pd.DataFrame(records)
agg["Net Required Qty"] = agg["Required Qty"] - agg["Inventory_25"]

# =============================
# TRACKING
# =============================
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)
machine_parts = defaultdict(set)

# =============================
# STEP 3: FOREMAN-AWARE LOAD BALANCING
# =============================
for _, row in agg.iterrows():

    child = row["Child Part"]
    net_qty = row["Net Required Qty"]
    daily_demand = row["Daily Demand"]

    if net_qty <= 0:
        continue

    ct = cycle_time_min.get(child)
    if not ct or ct <= 0:
        continue

    # 🧠 FOREMAN RULE
    foreman_cap = daily_demand * (1 + SAFE_BUFFER_DAYS)
    planned_qty = min(net_qty, foreman_cap)
    remaining_qty = planned_qty

    # Parse machines in priority order
    raw = str(row["Vertical Machines"])
    tokens = re.split(r"[,\|/\\\n]+", raw)

    eligible = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in eligible:
            eligible.append(m)

    if not eligible:
        continue

    for m in eligible:

        if remaining_qty <= 0:
            break

        available_time = MACHINE_CAPACITY - machine_load[m]

        # Changeover penalty (once per machine-part)
        if child not in machine_parts[m]:
            available_time -= CHANGEOVER_TIME
            if available_time <= 0:
                continue
            machine_load[m] += CHANGEOVER_TIME
            machine_parts[m].add(child)

        max_qty_fit = available_time / ct
        if max_qty_fit <= 0:
            continue

        assign_qty = min(remaining_qty, max_qty_fit)
        time_used = assign_qty * ct

        machine_load[m] += time_used
        remaining_qty -= assign_qty

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(assign_qty, 2),
            "Time Used (min)": round(time_used, 2)
        })

# =============================
# DISPLAY RESULTS
# =============================
print("\n========== MACHINE-WISE PLAN (FOREMAN LOGIC) ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No allocation")
    print("-" * 60)

print("\n========== MACHINE LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Capacity (min)": MACHINE_CAPACITY,
        "Utilization %": round((machine_load[m] / MACHINE_CAPACITY) * 100, 1)
    }
    for m in ALLOWED_MACHINES
]))


In [ ]:
import pandas as pd
import re
from collections import defaultdict

# =============================
# FILE PATH
# =============================
file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")

# =============================
# CLEAN NUMERIC COLUMNS
# =============================
for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =============================
# CYCLE TIME (SEC → MIN)
# =============================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 22 * 60          # 1320 min/day
CHANGEOVER_TIME = 40               # minutes
SAFE_BUFFER_DAYS = 1               # foreman rule

# =============================
# MACHINE NORMALIZER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: FILTER DAILY PLAN
# =============================
valid = master[master["Daily Plan"] > 0].copy()

# =============================
# STEP 2: AGGREGATION (CHILD PART)
# =============================
records = []

for child, grp in valid.groupby("Child Part", sort=False):

    daily_demand = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inventory = grp["Inventory_25"].iloc[0]
    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Daily Demand": daily_demand,
        "Required Qty": daily_demand + min_qty,
        "Inventory_25": inventory,
        "Vertical Machines": machines
    })

agg = pd.DataFrame(records)
agg["Net Required Qty"] = agg["Required Qty"] - agg["Inventory_25"]

# =============================
# TRACKING STRUCTURES
# =============================
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)
machine_parts = defaultdict(set)

unmet_parts = []
carry_forward = []

# =============================
# STEP 3: FOREMAN ASSIGNMENT
# =============================
for _, row in agg.iterrows():

    child = row["Child Part"]
    net_qty = row["Net Required Qty"]
    daily_demand = row["Daily Demand"]

    if net_qty <= 0:
        continue

    ct = cycle_time_min.get(child)
    if not ct or ct <= 0:
        continue

    # 🧠 FOREMAN CAP
    foreman_cap = daily_demand * (1 + SAFE_BUFFER_DAYS)
    planned_qty = min(net_qty, foreman_cap)
    remaining_qty = planned_qty

    # Parse machines
    raw = str(row["Vertical Machines"])
    tokens = re.split(r"[,\|/\\\n]+", raw)

    eligible = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in eligible:
            eligible.append(m)

    if not eligible:
        unmet_parts.append({
            "Child Part": child,
            "Unmet Qty": round(planned_qty, 2),
            "Reason": "No eligible machine"
        })
        carry_forward.append({
            "Child Part": child,
            "Carry Forward Qty": round(planned_qty, 2)
        })
        continue

    for m in eligible:

        if remaining_qty <= 0:
            break

        # ❌ skip fully utilized machines
        if machine_load[m] >= MACHINE_CAPACITY:
            continue

        available_time = MACHINE_CAPACITY - machine_load[m]

        # Changeover penalty (once per machine-part)
        if child not in machine_parts[m]:
            available_time -= CHANGEOVER_TIME
            if available_time <= 0:
                continue
            machine_load[m] += CHANGEOVER_TIME
            machine_parts[m].add(child)

        max_qty_fit = available_time / ct
        if max_qty_fit <= 0:
            continue

        assign_qty = min(remaining_qty, max_qty_fit)
        time_used = assign_qty * ct

        machine_load[m] += time_used
        remaining_qty -= assign_qty

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(assign_qty, 2),
            "Time Used (min)": round(time_used, 2)
        })

    # =============================
    # FLAG UNMET + CARRY FORWARD
    # =============================
    if remaining_qty > 0:
        unmet_parts.append({
            "Child Part": child,
            "Planned Qty": round(planned_qty, 2),
            "Produced Qty": round(planned_qty - remaining_qty, 2),
            "Unmet Qty": round(remaining_qty, 2)
        })

        carry_forward.append({
            "Child Part": child,
            "Carry Forward Qty": round(remaining_qty, 2)
        })

# =============================
# OUTPUTS
# =============================
print("\n========== MACHINE-WISE PLAN ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No allocation")
    print("-" * 60)

print("\n========== MACHINE LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Capacity (min)": MACHINE_CAPACITY,
        "Utilization %": round((machine_load[m] / MACHINE_CAPACITY) * 100, 1)
    }
    for m in ALLOWED_MACHINES
]))

print("\n========== ❌ UNMET PARTS ==========\n")
if unmet_parts:
    display(pd.DataFrame(unmet_parts))
else:
    print("✅ All planned quantities met")

print("\n========== 📦 CARRY FORWARD (TOMORROW) ==========\n")
if carry_forward:
    display(pd.DataFrame(carry_forward))
else:
    print("✅ No carry forward required")


In [ ]:
import pandas as pd
import re
from collections import defaultdict

# =============================
# FILE PATH
# =============================
file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")

# =============================
# CLEAN NUMERIC COLUMNS
# =============================
for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =============================
# CYCLE TIME (SEC → MIN)
# =============================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]  # 120T verticals
MACHINE_CAPACITY = 22 * 60          # 1320 min/day
CHANGEOVER_TIME = 40               # minutes
SAFE_BUFFER_DAYS = 1               # foreman rule

# =============================
# MACHINE NORMALIZER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# CHECK IF PART BELONGS TO 120T
# =============================
def is_120t_part(machine_str):
    raw = str(machine_str)
    tokens = re.split(r"[,\|/\\\n]+", raw)
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES:
            return True
    return False

# =============================
# STEP 1: FILTER DAILY PLAN
# =============================
valid = master[master["Daily Plan"] > 0].copy()

# =============================
# STEP 2: AGGREGATE (CHILD PART)
# =============================
records = []

for child, grp in valid.groupby("Child Part", sort=False):

    daily_demand = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inventory = grp["Inventory_25"].iloc[0]
    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Daily Demand": daily_demand,
        "Required Qty": daily_demand + min_qty,
        "Inventory_25": inventory,
        "Vertical Machines": machines
    })

agg = pd.DataFrame(records)
agg["Net Required Qty"] = agg["Required Qty"] - agg["Inventory_25"]

# =============================
# TRACKING STRUCTURES
# =============================
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)
machine_parts = defaultdict(set)

unmet_parts = []
carry_forward = []

# =============================
# STEP 3: FOREMAN ASSIGNMENT
# =============================
for _, row in agg.iterrows():

    child = row["Child Part"]
    net_qty = row["Net Required Qty"]
    daily_demand = row["Daily Demand"]

    if net_qty <= 0:
        continue

    ct = cycle_time_min.get(child)
    if not ct or ct <= 0:
        continue

    # 🧠 FOREMAN CAP (upper bound, not force)
    foreman_cap = daily_demand * (1 + SAFE_BUFFER_DAYS)
    planned_qty = min(net_qty, foreman_cap)
    remaining_qty = planned_qty

    # Parse eligible machines
    raw = str(row["Vertical Machines"])
    tokens = re.split(r"[,\|/\\\n]+", raw)

    eligible = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in eligible:
            eligible.append(m)

    if not eligible:
        unmet_parts.append({
            "Child Part": child,
            "Unmet Qty": round(planned_qty, 2),
            "Reason": "No 120T machine"
        })
        continue

    for m in eligible:

        if remaining_qty <= 0:
            break

        # ❌ skip fully utilized machines
        if machine_load[m] >= MACHINE_CAPACITY:
            continue

        available_time = MACHINE_CAPACITY - machine_load[m]

        # Changeover (once per machine-part)
        if child not in machine_parts[m]:
            available_time -= CHANGEOVER_TIME
            if available_time <= 0:
                continue
            machine_load[m] += CHANGEOVER_TIME
            machine_parts[m].add(child)

        max_qty_fit = available_time / ct
        if max_qty_fit <= 0:
            continue

        assign_qty = min(remaining_qty, max_qty_fit)
        time_used = assign_qty * ct

        machine_load[m] += time_used
        remaining_qty -= assign_qty

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(assign_qty, 2),
            "Time Used (min)": round(time_used, 2)
        })

    # =============================
    # FLAG UNMET + 120T CARRY FORWARD
    # =============================
    if remaining_qty > 0:
        unmet_parts.append({
            "Child Part": child,
            "Planned Qty": round(planned_qty, 2),
            "Produced Qty": round(planned_qty - remaining_qty, 2),
            "Unmet Qty": round(remaining_qty, 2)
        })

        # 📌 Carry-forward ONLY for 120T vertical parts
        if is_120t_part(row["Vertical Machines"]):
            carry_forward.append({
                "Child Part": child,
                "Carry Forward Qty": round(remaining_qty, 2),
                "Reason": "120T vertical capacity constraint"
            })

# =============================
# OUTPUTS
# =============================
print("\n========== MACHINE-WISE PLAN ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No allocation")
    print("-" * 60)

print("\n========== MACHINE LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Capacity (min)": MACHINE_CAPACITY,
        "Utilization %": round((machine_load[m] / MACHINE_CAPACITY) * 100, 1)
    }
    for m in ALLOWED_MACHINES
]))

print("\n========== ❌ UNMET PARTS ==========\n")
if unmet_parts:
    display(pd.DataFrame(unmet_parts))
else:
    print("✅ All planned quantities met")

print("\n========== 📦 CARRY FORWARD (120T ONLY) ==========\n")
if carry_forward:
    display(pd.DataFrame(carry_forward))
else:
    print("✅ No 120T carry-forward required")
